# 02 — Preprocessing & Feature Engineering

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np

## 1. Build master DataFrame

In [ ]:
from src.data_loader import build_master
master = build_master(download=True)
print(master.shape)
master.head(3)

## 2. Clean

In [ ]:
from src.preprocessing import clean, save_processed, save_country_splits
cleaned = clean(master)
print(cleaned.dtypes)
print(cleaned.isnull().sum().sort_values(ascending=False).head(10))

## 3. Feature engineering

In [ ]:
from src.features import build_features, get_feature_columns
featured = build_features(cleaned)
print("Feature columns:", get_feature_columns())
featured[['country','date'] + get_feature_columns()[:6]].head()

## 4. Save outputs

In [ ]:
save_processed(featured, 'features.csv')
save_country_splits(featured)
print("Done.")

## 5. Train / test split strategy
We use the last 30 days per country as the test set — a temporal holdout.

In [ ]:
from sklearn.model_selection import train_test_split

COUNTRIES = featured['country'].unique()
FEATURE_COLS = [c for c in get_feature_columns() if c in featured.columns]
TARGET = 'new_cases_7d'

def temporal_split(df, test_days=30):
    cutoff = df['date'].max() - pd.Timedelta(days=test_days)
    train = df[df['date'] <= cutoff]
    test  = df[df['date'] >  cutoff]
    return train, test

train, test = temporal_split(featured)
print(f"Train rows: {len(train):,}  |  Test rows: {len(test):,}")
print(f"Train ends: {train.date.max().date()}  Test starts: {test.date.min().date()}")